# Chapter 9 -- Agent Memory (Practice)

Work through this notebook **after reading** `notes/ch09-agent-memory.md`. This chapter builds a real three-scope memory system from scratch on SQLite -- a hot-path writer, a deterministic cold-path fact extractor, a salience-based retriever matching notes Section 12's formula exactly, and supersession-based contradiction handling -- then runs it against a genuine 20-turn scripted conversation, printing every write. It closes by comparing recall against a flat, Mem0-style store with no supersession, to make Section 7/8's argument concrete rather than asserted.

Three exercises below have a stub to fill in: **decay** (Section 8), **supersession** (Section 8), and **a 10-item memory regression suite** (Section 9). Everything is fully offline and deterministic -- no API key needed for any exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- The Three-Scope Schema (Given)

Notes Section 2: episodic (raw, append-only, timestamped turns), semantic (subject/predicate/object facts, with an `active` flag for supersession), procedural (rules, not exercised numerically in this notebook but included for completeness). All three live in one SQLite database, in memory, for a clean deterministic run.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE episodic (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        turn_index INTEGER,
        text TEXT,
        consolidated INTEGER DEFAULT 0
    )
""")
conn.execute("""
    CREATE TABLE semantic (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        subject TEXT,
        predicate TEXT,
        object TEXT,
        importance REAL,
        turn_index INTEGER,
        active INTEGER DEFAULT 1
    )
""")
conn.execute("""
    CREATE TABLE procedural (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        rule_text TEXT,
        turn_index INTEGER
    )
""")
conn.commit()
print("Schema created: episodic, semantic, procedural.")


def hot_path_write(conn, turn_index, text):
    """Synchronous, cheap, every turn -- notes Section 5's hot path. Just an append."""
    conn.execute("INSERT INTO episodic (turn_index, text, consolidated) VALUES (?, ?, 0)", (turn_index, text))
    conn.commit()
    print(f"  [hot path] wrote turn {turn_index}: {text!r}")


## Part 2 -- Deterministic Fact Extraction and the Cold Path (Given)

A real cold-path consolidator (notes Section 5) would use an LLM to extract facts; this notebook uses simple, deterministic regexes instead, specifically so the whole run is reproducible with no API key. The five fact types below are worded to match notes Section 12's five dry-run memories almost exactly, on purpose -- so Part 3 can verify the salience formula against real extracted facts, not just the notes' hand-picked table.

In [ ]:
import re

LANG_RE = re.compile(r"prefers? (\w+) over (\w+)", re.I)
MANAGER_RE = re.compile(r"manager is (\w+)", re.I)
DB_RE = re.compile(r"uses (\w+) for the main database", re.I)
DEADLINE_RE = re.compile(r"deadline is (\w+)", re.I)
BEVERAGE_RE = re.compile(r"i like (\w+), not (\w+)", re.I)


def extract_facts(text):
    """Return a list of (subject, predicate, object, importance) tuples found in `text`."""
    facts = []
    if m := LANG_RE.search(text):
        facts.append(("user", "prefers_language", m.group(1), 0.6))
    if m := MANAGER_RE.search(text):
        facts.append(("user", "manager", m.group(1), 0.9))
    if m := DB_RE.search(text):
        facts.append(("project", "database", m.group(1), 0.7))
    if m := DEADLINE_RE.search(text):
        facts.append(("project", "deadline", m.group(1), 0.8))
    if m := BEVERAGE_RE.search(text):
        facts.append(("user", "beverage_preference", m.group(1), 0.2))
    return facts


print("extract_facts defined. Quick check:")
for sample in ["I prefer Python over JavaScript for backend work.", "My manager is Priya, she reviews all my PRs."]:
    print(f"  {sample!r} -> {extract_facts(sample)}")


In [ ]:
def naive_upsert_semantic(conn, subject, predicate, object_, importance, turn_index):
    """
    The PROBLEM this section deliberately leaves in place: always INSERT,
    never check for an existing active fact with the same (subject, predicate).
    Exercise 2 replaces this with real supersession.
    """
    conn.execute(
        "INSERT INTO semantic (subject, predicate, object, importance, turn_index, active) VALUES (?, ?, ?, ?, ?, 1)",
        (subject, predicate, object_, importance, turn_index),
    )
    conn.commit()


def cold_path_consolidate(conn, upsert_fn=naive_upsert_semantic):
    """
    Asynchronous, batched, expensive-relative-to-hot-path -- notes Section 5's
    cold path. Scans every not-yet-consolidated episodic row, extracts facts,
    and upserts each one using whichever upsert function is passed in.
    """
    rows = conn.execute("SELECT id, turn_index, text FROM episodic WHERE consolidated = 0").fetchall()
    n_facts = 0
    for row_id, turn_index, text in rows:
        for subject, predicate, object_, importance in extract_facts(text):
            upsert_fn(conn, subject, predicate, object_, importance, turn_index)
            n_facts += 1
            print(f"  [cold path] turn {turn_index}: extracted ({subject}, {predicate}, {object_}) importance={importance}")
        conn.execute("UPDATE episodic SET consolidated = 1 WHERE id = ?", (row_id,))
    conn.commit()
    return n_facts


print("naive_upsert_semantic and cold_path_consolidate defined.")


## Part 3 -- Salience Scoring, Verified Against Notes Section 12

The exact formula from the notes, and the exact five memories from the notes' dry-run table -- computed here in code to confirm the by-hand rankings at both decay rates, including the reordering at $\lambda=1.0$.

In [ ]:
import math

def salience(sim, delta_t_days, importance, alpha=0.5, beta=0.3, gamma=0.2, lam=0.05):
    """s = alpha*sim + beta*e^(-lambda*delta_t) + gamma*importance -- notes Section 12."""
    return alpha * sim + beta * math.exp(-lam * delta_t_days) + gamma * importance


NOTES_MEMORIES = {
    "M1": dict(sim=0.90, delta_t_days=2,  importance=0.6),
    "M2": dict(sim=0.40, delta_t_days=30, importance=0.9),
    "M3": dict(sim=0.70, delta_t_days=1,  importance=0.8),
    "M4": dict(sim=0.30, delta_t_days=60, importance=0.2),
    "M5": dict(sim=0.85, delta_t_days=15, importance=0.7),
}

for lam, expected_order in [(0.05, ["M1", "M3", "M5", "M2", "M4"]), (1.0, ["M3", "M1", "M5", "M2", "M4"])]:
    scores = {name: salience(**vals, lam=lam) for name, vals in NOTES_MEMORIES.items()}
    ranked = sorted(scores, key=lambda name: -scores[name])
    print(f"lambda={lam}: " + "  ".join(f"{name}={scores[name]:.4f}" for name in ranked))
    assert ranked == expected_order, f"expected order {expected_order} at lambda={lam}, got {ranked}"

print("\nBoth rankings match notes Section 12 exactly, including the M1/M3 swap at lambda=1.0.")


## Part 4 -- Running a Real 20-Turn Conversation Through the Hot Path

Every turn is written via `hot_path_write` (printed as it happens), then the cold path consolidates all of them at once, using the still-naive `naive_upsert_semantic` -- deliberately, so the contradiction problem below is real and visible, not asserted. Turn 13 changes the user's language preference from Python to Rust; watch what that does to the semantic table with no supersession logic in place yet.

In [ ]:
CONVERSATION = [
    "Hi, I'm working on the checkout redesign project.",
    "I prefer Python over JavaScript for backend work.",
    "Can you help me set up a test environment?",
    "My manager is Priya, she reviews all my PRs.",
    "Let's talk about the database schema next.",
    "Our project uses PostgreSQL for the main database.",
    "What's a good way to handle migrations?",
    "I like coffee, not tea, if you're ever making a round.",
    "The deadline is Friday for this sprint.",
    "Can we go over the API design once more?",
    "I think the error handling needs work.",
    "Let's add more logging to the payment service.",
    "Actually, I prefer Rust over JavaScript now for backend work.",
    "That refactor looks solid, thanks.",
    "Can you check the test coverage numbers?",
    "I want to revisit the caching strategy.",
    "Let's schedule a sync with the frontend team.",
    "The staging environment needs a redeploy.",
    "Good progress today, let's continue tomorrow.",
    "One more thing -- can you summarize what we covered?",
]

print("-" * 60)
print("HOT PATH: writing all 20 turns")
print("-" * 60)
for i, text in enumerate(CONVERSATION, start=1):
    hot_path_write(conn, i, text)

print()
print("-" * 60)
print("COLD PATH: consolidating (naive upsert -- no supersession yet)")
print("-" * 60)
n_facts = cold_path_consolidate(conn, upsert_fn=naive_upsert_semantic)
print(f"\n{n_facts} facts extracted across the conversation.")


In [ ]:
print("Active semantic facts for (user, prefers_language):")
rows = conn.execute(
    "SELECT object, turn_index, active FROM semantic WHERE subject='user' AND predicate='prefers_language' AND active=1"
).fetchall()
for object_, turn_index, active in rows:
    print(f"  object={object_!r}  written at turn {turn_index}  active={active}")

print(f"\n{len(rows)} active rows for the SAME (subject, predicate) -- this is exactly notes")
print("Section 8's warning made concrete: a retriever asking 'what language does the user")
print("prefer' has no principled way to choose between Python and Rust from this table alone.")
assert len(rows) == 2, "expected the naive upsert to leave BOTH Python and Rust active -- that's the bug Exercise 2 fixes"


## Exercise 1 -- Decay

Implement `decayed_importance(importance, delta_t_days, half_life_days=30)`: notes Section 8's softer cousin to a hard TTL. A memory's importance should be cut in half every `half_life_days`, continuously, rather than dropping to zero at a fixed cutoff. Use `importance * 0.5 ** (delta_t_days / half_life_days)`.

In [ ]:
def decayed_importance(importance, delta_t_days, half_life_days=30):
    """Importance halves every half_life_days, continuously -- notes Section 8."""
    # TODO: return importance scaled by 0.5 ** (delta_t_days / half_life_days).
    # At delta_t_days=0 this should return importance unchanged; at
    # delta_t_days == half_life_days it should return exactly half.
    return importance


In [ ]:
# At delta_t=0, decay should have no effect at all.
assert abs(decayed_importance(0.9, 0) - 0.9) < 1e-9

# At delta_t == half_life, importance should be exactly halved.
assert abs(decayed_importance(0.8, 30, half_life_days=30) - 0.4) < 1e-9

# At delta_t == 2 * half_life, importance should be a quarter of the original.
assert abs(decayed_importance(0.8, 60, half_life_days=30) - 0.2) < 1e-9

# A shorter half-life should decay faster at the same delta_t.
assert decayed_importance(0.8, 30, half_life_days=15) < decayed_importance(0.8, 30, half_life_days=30)

print("Exercise 1 PASSED -- decayed_importance halves correctly at one and two half-lives,")
print("leaves brand-new memories untouched, and decays faster with a shorter half-life.")


## Exercise 2 -- Supersession

Implement `supersede_and_upsert(conn, subject, predicate, object_, importance, turn_index)`: notes Section 8's real fix for the problem Part 4 just demonstrated. Before inserting the new fact, find any existing **active** row with the same `(subject, predicate)` but a **different** `object` value, and mark it `active = 0` (superseded, not deleted -- its history stays queryable). Then insert the new fact as active. If an active row already has the exact same `object` (nothing actually changed), leave it alone and don't insert a duplicate.

In [ ]:
def supersede_and_upsert(conn, subject, predicate, object_, importance, turn_index):
    """
    Mark any existing active (subject, predicate) row with a DIFFERENT object
    as superseded, then insert the new fact as active. No-op if the exact
    same fact is already active.
    """
    # TODO:
    # 1. SELECT id, object FROM semantic WHERE subject=? AND predicate=? AND active=1
    #    (parameterized with subject, predicate) into `existing`.
    # 2. For each (row_id, existing_object) in existing: if existing_object
    #    equals the new object_, this fact hasn't actually changed -- return
    #    immediately, inserting nothing. Otherwise, UPDATE that row to active=0
    #    (superseded, not deleted).
    # 3. After the loop, INSERT the new fact as an active row and commit.
    existing = []
    # your code here

    conn.commit()


In [ ]:
# Rebuild the memory system from scratch with the real supersession logic,
# so the final state reflects supersede_and_upsert throughout, not the naive version.
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE episodic (id INTEGER PRIMARY KEY AUTOINCREMENT, turn_index INTEGER, text TEXT, consolidated INTEGER DEFAULT 0)")
conn.execute("CREATE TABLE semantic (id INTEGER PRIMARY KEY AUTOINCREMENT, subject TEXT, predicate TEXT, object TEXT, importance REAL, turn_index INTEGER, active INTEGER DEFAULT 1)")

print("-" * 60)
print("RE-RUNNING THE FULL CONVERSATION WITH REAL SUPERSESSION")
print("-" * 60)
for i, text in enumerate(CONVERSATION, start=1):
    hot_path_write(conn, i, text)
n_facts = cold_path_consolidate(conn, upsert_fn=supersede_and_upsert)

active_lang = conn.execute(
    "SELECT object, turn_index FROM semantic WHERE subject='user' AND predicate='prefers_language' AND active=1"
).fetchall()
superseded_lang = conn.execute(
    "SELECT object, turn_index FROM semantic WHERE subject='user' AND predicate='prefers_language' AND active=0"
).fetchall()

print(f"\nActive prefers_language facts: {active_lang}")
print(f"Superseded prefers_language facts: {superseded_lang}")

assert active_lang == [("Rust", 13)], f"expected exactly Rust from turn 13 active, got {active_lang}"
assert superseded_lang == [("Python", 2)], f"expected Python from turn 2 superseded, got {superseded_lang}"
print("\nExercise 2 PASSED -- Rust (the newer fact) is the sole active row; Python is")
print("superseded, not deleted -- its history is still there if anything ever needs it.")


## Given: Comparing Against a Flat, Mem0-Style Store With No Supersession

`FlatSemanticStore` is a deliberately simplified stand-in for the *architectural pattern* a flat semantic-only store represents (notes Section 6) -- **not** the real Mem0 SDK, which does implement its own contradiction handling internally. The point here is narrower and purely illustrative: a store with no notion of "this new fact replaces that old one" has to return *everything* that matches a query, ambiguity and all, which is exactly notes Section 7's contradiction-handling argument made concrete.

In [ ]:
class FlatSemanticStore:
    """Append-only, no active/inactive distinction, no supersession -- a stand-in for a flat store."""

    def __init__(self):
        self.facts = []

    def write(self, subject, predicate, object_, turn_index):
        self.facts.append((subject, predicate, object_, turn_index))

    def recall(self, subject, predicate):
        return [f for f in self.facts if f[0] == subject and f[1] == predicate]


flat_store = FlatSemanticStore()
for i, text in enumerate(CONVERSATION, start=1):
    for subject, predicate, object_, importance in extract_facts(text):
        flat_store.write(subject, predicate, object_, i)

flat_result = flat_store.recall("user", "prefers_language")
scoped_result = conn.execute(
    "SELECT subject, predicate, object, turn_index FROM semantic WHERE subject='user' AND predicate='prefers_language' AND active=1"
).fetchall()

print("-" * 60)
print("RECALL COMPARISON: 'what language does the user prefer?'")
print("-" * 60)
print(f"Flat store (no supersession):        {flat_result}")
print(f"Three-scope store (with supersession): {scoped_result}")

assert len(flat_result) == 2, "the flat store should still show BOTH Python and Rust -- it has no way to resolve this"
assert len(scoped_result) == 1 and scoped_result[0][2] == "Rust"
print("\nConfirmed: the flat store returns both, ambiguously, with no signal about which is")
print("current. The three-scope store with supersession returns exactly one, correctly.")


## Exercise 3 -- A 10-Item Memory Regression Suite

Notes Section 9: a small, hand-built regression set catches failures a general benchmark can't anticipate. Fill in `REGRESSION_CASES` with **10** `(description, check_fn)` pairs, where each `check_fn` takes the `conn` connection and returns `True`/`False`. Cover at least: each of the 5 active facts having the right value, the superseded Python fact being correctly inactive, a query for a fact that was never mentioned correctly returning nothing, the episodic row count, and the count of currently-active semantic facts.

In [ ]:
def get_active(conn, subject, predicate):
    row = conn.execute(
        "SELECT object FROM semantic WHERE subject=? AND predicate=? AND active=1", (subject, predicate)
    ).fetchone()
    return row[0] if row else None


# TODO: fill in exactly 10 (description, check_fn) pairs. Each check_fn takes
# `conn` and returns True/False. Use `get_active(conn, subject, predicate)`
# for active-fact lookups, and raw conn.execute(...).fetchone() queries for
# the rest. Cover: each of the 5 active facts having the right value, the
# superseded Python fact being correctly inactive, a never-mentioned fact
# returning None, the episodic row count, the consolidated-flag count, and
# the count of currently-active semantic facts.
REGRESSION_CASES = [
    # your 10 cases here
]


In [ ]:
def run_regression_suite(conn, cases):
    passed, failed = [], []
    for description, check_fn in cases:
        ok = check_fn(conn)
        (passed if ok else failed).append(description)
        print(f"  {'PASS' if ok else 'FAIL'}: {description}")
    return passed, failed


print("-" * 60)
print(f"MEMORY REGRESSION SUITE ({len(REGRESSION_CASES)} cases)")
print("-" * 60)
passed, failed = run_regression_suite(conn, REGRESSION_CASES)

assert len(REGRESSION_CASES) == 10, f"expected exactly 10 regression cases, got {len(REGRESSION_CASES)}"
assert not failed, f"expected all cases to pass, but these failed: {failed}"
print(f"\nExercise 3 PASSED -- all {len(passed)}/10 regression cases pass against the final,")
print("supersession-corrected memory state.")


## Optional -- Have a Real Claude Model Extract a Semantic Fact

The same extraction job Part 2 did with regexes, but asked of a real model instead -- a genuinely live comparison of the deterministic stand-in against the thing it's standing in for.

In [ ]:
RUN_REAL_EXTRACTION_DEMO = False


def run_real_extraction_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real extraction demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    prompt = (
        "Extract any durable semantic fact from this sentence as a (subject, predicate, object) "
        "triple, or say NONE if there isn't one: "
        "\"Actually, I prefer Rust over JavaScript now for backend work.\""
    )
    try:
        response = real_client.messages.create(
            model=MODEL_NAME, max_tokens=100,
            messages=[{"role": "user", "content": prompt}],
        )
        text = next((b.text for b in response.content if b.type == "text"), "")
        print(text)
    except Exception as exc:
        print(f"Real extraction demo failed: {type(exc).__name__}: {exc}")


if RUN_REAL_EXTRACTION_DEMO:
    run_real_extraction_demo()
else:
    print("RUN_REAL_EXTRACTION_DEMO is False -- running in offline/regex mode only.")
    print("Flip it to True to compare against a real Claude model's own extraction via Bedrock.")


## Key Takeaways

You built all three memory scopes on real SQLite tables, verified the salience formula against notes Section 12's exact numbers (including the lambda-driven reordering), and watched a real contradiction emerge from the naive upsert before fixing it with supersession -- Python correctly demoted to inactive history, Rust correctly the sole active fact. The flat-store comparison made Section 7's abstract argument concrete: a store with no supersession model has no way to resolve the exact same ambiguity your three-scope system resolved cleanly. The regression suite is small, but it's the same idea Chapter 14 will scale up to the whole harness: a handful of checks, re-run after every change, catching exactly the failures a general benchmark has no way to anticipate.

**Connection forward:** Chapter 10 picks up where the `FlatSemanticStore` comparison left off -- once memory needs to represent relationships between facts (not just facts in isolation), even a supersession-aware flat table stops being enough, and the question becomes when a graph actually earns the complexity it adds.